In [85]:
import os
import csv
import time
from llama_cpp import Llama

In [86]:
from config import (
    MODEL_CTX,
    GPU_LAYERS,
    INPUT_PATH,
    OUTPUT_FILE,
    INPUT_DIR,
    INPUT_FILENAME,
    MAX_TOKENS_CHUNK,
    MAX_TOKENS_FINAL,
    TEMPERATURE,
    CHUNK_SIZE,
    CHUNK_OVERLAP
)

MODEL_PATH = r'Bielik-1.5B-v3.0-Instruct.Q8_0.gguf'
print(MODEL_PATH)

Bielik-1.5B-v3.0-Instruct.Q8_0.gguf


In [87]:
# ═══════════════════════════════════════��══════════════════════
# INICJALIZACJA MODELU
# ══════════════════════════════════════════════════════════════
def load_model():
    """Ładuje model LLM i zwraca instancję."""
    if not os.path.exists(MODEL_PATH):
        print(MODEL_PATH)
        raise FileNotFoundError(f"Model nie znaleziony: {MODEL_PATH}")

    print(f"[INFO] Ładowanie modelu: {os.path.basename(MODEL_PATH)}")
    start = time.time()

    llm = Llama(
        model_path=MODEL_PATH,
        n_ctx=MODEL_CTX,
        n_gpu_layers=GPU_LAYERS,
        verbose=False,
    )

    elapsed = time.time() - start
    print(f"[INFO] Model załadowany w {elapsed:.1f}s")
    return llm

In [88]:
# ══════════════════════════════════════════════════════════════
# FORMATOWANIE PROMPTU (ChatML — format Bielika)
# ══════════════════════════════════════════════════════════════
def create_prompt(system: str, user: str) -> str:
    """Tworzy prompt w formacie ChatML używanym przez Bielik-Instruct."""
    return (
        f"<|im_start|>system\n{system}<|im_end|>\n"
        f"<|im_start|>user\n{user}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


In [89]:
# ═════════════════════════════════════════════��════════════════
# DZIELENIE TEKSTU NA FRAGMENTY
# ══════════════════════════════════════════════════════════════
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE,
               overlap: int = CHUNK_OVERLAP) -> list[str]:
    """
    Dzieli tekst na nakładające się fragmenty.
    Stara się dzielić na granicy zdania (kropka/średnik).
    """
    text = text.strip()
    if not text:
        return []

    # Krótki tekst — nie trzeba dzielić
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        # Próba znalezienia granicy zdania (szukamy w ostatnich 20% fragmentu)
        if end < len(text):
            search_start = max(start, end - chunk_size // 5)
            last_period = text.rfind('.', search_start, end)
            last_semicolon = text.rfind(';', search_start, end)
            best_break = max(last_period, last_semicolon)

            if best_break > start:
                end = best_break + 1  # +1 żeby zostawić kropkę

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        # Przesunięcie z uwzględnieniem nakładki
        if end >= len(text):
            break
        start = end - overlap

    return chunks

In [90]:
# ══════════════════════════════════════════════════════════════
# ZAPYTANIE DO LLM
# ══════════════════════════════════════════════════════════════
def get_llm_response(llm, prompt: str, max_tokens: int = 256,
                     temperature: float = 0.1) -> str:
    """Wysyła prompt do modelu i zwraca odpowiedź."""
    try:
        output = llm(
            prompt,
            max_tokens=max_tokens,
            stop=["<|im_end|>"],
            temperature=temperature,
            repeat_penalty=1.1,  # Zmniejsza powtórzenia
        )
        return output['choices'][0]['text'].strip()
    except Exception as e:
        print(f"    [WARN] Błąd LLM: {e}")
        return "BŁĄD_GENEROWANIA"

In [91]:
# ══════════════════════════════════════════════════════════════
# PROMPTY SYSTEMOWE I UŻYTKOWNIKA
# ══════════════════════════════════════���═══════════════════════
# --- PROMPTY ---

SYSTEM_EXTRACT = "Wyodrębnij branżę, produkty i usługi z opisu firmy."

USER_EXTRACT_TEMPLATE = (
    "Wypisz dane z tekstu w formacie:\n"
    "Branża: [nazwa]\n"
    "Produkty: [max 5, lub 'brak']\n"
    "Usługi: [max 5, lub 'brak']\n\n"
    "Pomijaj adresy i lokalizacje.\n"
    "Jeśli tekst pusty lub 'BRAK OPISU' napisz tylko: BRAK OPISU\n\n"
    "Tekst:\n{text}"
)

SYSTEM_MERGE = "Scal dane w jeden wpis. Usuń duplikaty."

USER_MERGE_TEMPLATE = (
    "Scal w format: Branża: ... | Produkty: max 5 | Usługi: max 5\n"
    "Usuń powtórzenia.\n\n"
    "Dane:\n{fragments}"
)


In [92]:
# ══════════════════════════════════════════════════════════��═══
# ANALIZA JEDNEGO OPISU (MAP-REDUCE)
# ══════════════════════════════════════════════════════════════
def analyze_single_text(llm, text: str) -> str:
    """
    Przetwarza jeden opis firmy:
    1. Dzieli na fragmenty (jeśli długi)
    2. Analizuje każdy fragment (MAP)
    3. Scala wyniki (REDUCE)
    """
    text = text.strip()

    # Szybkie sprawdzenia
    if not text:
        return "BRAK TREŚCI"

    if text.upper() in ("BRAK OPISU", "BRAK", "-", ""):
        return "BRAK OPISU"

    # Podział na fragmenty
    chunks = chunk_text(text)

    if not chunks:
        return "BRAK TREŚCI"

    # ── KROK 1: Analiza fragmentów (MAP) ──
    partial_results = []

    for idx, chunk in enumerate(chunks, 1):
        if len(chunks) > 1:
            print(f"    Fragment {idx}/{len(chunks)}...")

        user_prompt = USER_EXTRACT_TEMPLATE.format(text=chunk)
        prompt = create_prompt(SYSTEM_EXTRACT, user_prompt)
        result = get_llm_response(
            llm, prompt,
            max_tokens=MAX_TOKENS_CHUNK,
            temperature=TEMPERATURE
        )

        if result and result != "BŁĄD_GENEROWANIA":
            partial_results.append(result)

    if not partial_results:
        return "BRAK WYNIKÓW"

    # ── KROK 2: Scalanie (REDUCE) ──
    if len(partial_results) == 1:
        final_result = partial_results[0]
    else:
        combined = "\n---\n".join(partial_results)
        # Ograniczenie długości, żeby zmieścić się w kontekście
        combined = combined[:2500]

        user_prompt = USER_MERGE_TEMPLATE.format(fragments=combined)
        prompt = create_prompt(SYSTEM_MERGE, user_prompt)

        final_result = get_llm_response(
            llm, prompt,
            max_tokens=MAX_TOKENS_FINAL,
            temperature=0.2
        )

    # ── Czyszczenie wyniku pod CSV ──
    clean = final_result.replace('\n', ' | ').replace('\r', '')
    clean = clean.replace(';', ',')  # Żeby nie łamać CSV
    clean = ' '.join(clean.split())  # Usunięcie podwójnych spacji

    return clean

In [93]:
# ══════════════════════════════════════════════════════════════
# PRZETWARZANIE PLIKU
# ══════════════════════════════════════════════════════════════
def count_lines(filepath: str) -> int:
    """Zlicza liczbę linii w pliku (do paska postępu)."""
    with open(filepath, 'r', encoding='utf-8') as f:
        return sum(1 for _ in f)


def count_processed_lines(output_path: str) -> int:
    """Zlicza już przetworzone linie (do wznowienia)."""
    if not os.path.exists(output_path):
        return 0
    with open(output_path, 'r', encoding='utf-8-sig') as f:
        return sum(1 for line in f if line.strip())


def process_file():
    """Główna funkcja przetwarzania pliku."""

    # ── Walidacja ──
    if not os.path.exists(INPUT_PATH):
        print(f"[BŁĄD] Plik wejściowy nie istnieje: {INPUT_PATH}")
        if not os.path.exists(INPUT_DIR):
            os.makedirs(INPUT_DIR)
            print(f"[INFO] Utworzono katalog '{INPUT_DIR}'. "
                  f"Umieść tam plik '{INPUT_FILENAME}'.")
        return

    # ── Ładowanie modelu ──
    llm = load_model()

    # ── Liczenie linii ──
    total_lines = count_lines(INPUT_PATH)
    already_done = count_processed_lines(OUTPUT_FILE)

    print(f"[INFO] Plik wejściowy: {INPUT_PATH} ({total_lines} linii)")
    print(f"[INFO] Plik wynikowy:  {OUTPUT_FILE}")

    if already_done > 0:
        print(f"[INFO] Wznawiam od linii {already_done + 1} "
              f"(już przetworzono: {already_done})")

    # ── Przetwarzanie ──
    processed = 0
    errors = 0
    start_time = time.time()

    # Tryb dopisywania ('a') żeby umożliwić wznowienie
    write_mode = 'a' if already_done > 0 else 'w'

    with open(INPUT_PATH, 'r', encoding='utf-8') as f_in, \
         open(OUTPUT_FILE, write_mode, encoding='utf-8-sig') as f_out:

        # Nagłówek (tylko przy nowym pliku)
        if write_mode == 'w':
            f_out.write("DANE_ORYGINALNE;WYNIK_AI\n")
            f_out.flush()

        for i, line in enumerate(f_in, 1):
            line = line.strip()

            # Pomiń puste linie
            if not line:
                continue

            # Pomiń już przetworzone (wznowienie)
            if i <= already_done:
                continue

            # ── Pasek postępu ──
            elapsed = time.time() - start_time
            if processed > 0:
                avg_time = elapsed / processed
                remaining = avg_time * (total_lines - i)
                eta = f"~{remaining/60:.0f}min"
            else:
                eta = "obliczam..."

            print(f"\n{'='*60}")
            print(f"[{i}/{total_lines}] Przetwarzanie... (ETA: {eta})")
            print(f"  Tekst: {line[:80]}{'...' if len(line)>80 else ''}")

            try:
                # Analiza przez LLM
                line_start = time.time()
                ai_result = analyze_single_text(llm, line)
                line_time = time.time() - line_start

                # Zapis
                output_line = f"{line};{ai_result}\n"
                f_out.write(output_line)
                f_out.flush()

                processed += 1
                print(f"  [OK] Czas: {line_time:.1f}s")
                print(f"  Wynik: {ai_result[:100]}{'...' if len(ai_result)>100 else ''}")

            except KeyboardInterrupt:
                print(f"\n[STOP] Przerwano przez użytkownika po {processed} liniach.")
                print(f"[INFO] Możesz wznowić — skrypt pominie przetworzone linie.")
                return

            except Exception as e:
                errors += 1
                print(f"  [BŁĄD] {type(e).__name__}: {e}")
                f_out.write(f"{line};BŁĄD: {str(e)[:50]}\n")
                f_out.flush()

    # ── Podsumowanie ──
    total_time = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"[KONIEC] Przetwarzanie zakończone!")
    print(f"  Przetworzono:  {processed} linii")
    print(f"  Błędy:         {errors}")
    print(f"  Łączny czas:   {total_time/60:.1f} min")
    if processed > 0:
        print(f"  Średni czas:   {total_time/processed:.1f}s / linia")
    print(f"  Wynik w:       {OUTPUT_FILE}")

In [94]:
# ══════════════════════════════════════════════════════════════
# URUCHOMIENIE
# ══════════════════════════════════════════════════════════════
if __name__ == "__main__":
    process_file()

[INFO] Ładowanie modelu: Bielik-1.5B-v3.0-Instruct.Q8_0.gguf


llama_context: n_ctx_seq (4096) < n_ctx_train (8192) -- the full capacity of the model will not be utilized


[INFO] Model załadowany w 0.3s
[INFO] Plik wejściowy: /home/onyxia/work/WP10-CLuster_2-example_code/Summary/input/input.csv (7 linii)
[INFO] Plik wynikowy:  /home/onyxia/work/WP10-CLuster_2-example_code/Summary/output/WP-10_gotowe.csv
[INFO] Wznawiam od linii 8 (już przetworzono: 7)

[KONIEC] Przetwarzanie zakończone!
  Przetworzono:  0 linii
  Błędy:         0
  Łączny czas:   0.0 min
  Wynik w:       /home/onyxia/work/WP10-CLuster_2-example_code/Summary/output/WP-10_gotowe.csv
